# Section 1: Load Libraries and Convert to CSV
Import necessary libraries (`pandas`, `geopy`, `openpyxl`). Read the provided `CSR_activities_2014-15.xlsx` file and convert it to a CSV file for easier processing.

In [1]:
import pandas as pd
# Install missing dependencies if needed:
# !pip install pandas openpyxl geopy

# 1. Load the dataset from Excel
excel_path = 'CSR_activities_2014-15.xlsx'
csv_path = 'CSR_activities_2014-15.csv'

try:
    df = pd.read_excel(excel_path)
    print(f"Successfully loaded {excel_path}")
    
    # 2. Convert and save to CSV
    df.to_csv(csv_path, index=False)
    print(f"Successfully saved to {csv_path}")
except FileNotFoundError:
    print(f"File not found: {excel_path}. Please ensure it is in the same directory.")
    df = pd.DataFrame() # create empty dataframe so later cells don't crash


Successfully loaded CSR_activities_2014-15.xlsx
Successfully saved to CSR_activities_2014-15.csv


# Section 2: Preliminary Exploratory Data Analysis (EDA)
Load the newly created CSV, inspect the dataframe using `head()`, `info()`, `describe()`, and check for missing values to understand the dataset structure.

In [2]:
# Load the created CSV
if not df.empty:
    df_csv = pd.read_csv(csv_path)

    # Output basic dataset structure
    print("--- DataFrame Info ---")
    df_csv.info()

    display(df_csv.head())

    print("\n--- Summary Statistics ---")
    display(df_csv.describe(include='all'))

    print("\n--- Missing Values ---")
    print(df_csv.isnull().sum())
else:
    print("Dataframe is empty, EDA skipped.")

--- DataFrame Info ---
<class 'pandas.DataFrame'>
RangeIndex: 5136 entries, 0 to 5135
Data columns (total 12 columns):
 #   Column                                              Non-Null Count  Dtype  
---  ------                                              --------------  -----  
 0   Symbol                                              5136 non-null   str    
 1   Company                                             5136 non-null   str    
 2   Activity                                            5136 non-null   str    
 3   Sector                                              204 non-null    str    
 4   Schedule                                            5136 non-null   str    
 5   Amount to be Spent as decided by Company (Rs.lacs)  4537 non-null   float64
 6   Actual Amount Spent (Rs.lacs)                       5065 non-null   float64
 7   Direct-Agency Name/s                                537 non-null    str    
 8   Direct-Amount (Rs.lacs)                             2842 non-null 

,Symbol,Company,Activity,Sector,Schedule,Amount to be Spent as decided by Company (Rs.lacs),Actual Amount Spent (Rs.lacs),Direct-Agency Name/s,Direct-Amount (Rs.lacs),Implementing Agency Name/s,Implementing Agency-Amount (Rs.lacs),States (City/Town/District/Village)
0,3MINDIA,3M INDIA LTD.,DONATION TOWARDS JAMMU & KASHMIR DISASTER RELI...,NaN,SCHEDULE VII (VIII),40.0,5.00,NaN,2.5000,NaN,2.5000,JAMMU & KASHMIR
1,3MINDIA,3M INDIA LTD.,EDUCATION: MOBILE SCIENCE LAB INITIATIVE FOR D...,NaN,SCHEDULE VII (II),48.0,44.41,NaN,22.2065,NaN,22.2065,"KARNATAKA (ANEKAL),MAHARASHTRA (RANJANGAON, PUNE)"
2,63MOONS,63 MOONS TECHNOLOGIES LTD.,A) DN POLYTECHNIC EDUCATION TR UST (ENTREPRENE...,NaN,SCHEDULE VII (II),NaN,245.00,NaN,122.5000,NaN,122.5000,ALL STATES
3,63MOONS,63 MOONS TECHNOLOGIES LTD.,B) SHRIMATI SAVITA MAHILA UTKARSH SEVA SANGH (...,NaN,SCHEDULE VII (II),NaN,30.00,NaN,15.0000,NaN,15.0000,ALL STATES
4,63MOONS,63 MOONS TECHNOLOGIES LTD.,C) JAYBHARTI FOUNDATION 50.00 (HEALTH AWARENES...,NaN,"SCHEDULE VII (I), SCHEDULE VII (III)",NaN,50.00,NaN,25.0000,NaN,25.0000,ALL STATES



--- Summary Statistics ---


,Symbol,Company,Activity,Sector,Schedule,Amount to be Spent as decided by Company (Rs.lacs),Actual Amount Spent (Rs.lacs),Direct-Agency Name/s,Direct-Amount (Rs.lacs),Implementing Agency Name/s,Implementing Agency-Amount (Rs.lacs),States (City/Town/District/Village)
count,5136,5136,5136,204,5136,4537.000000,5065.000000,537,2842.000000,1621,2894.000000,5098
unique,801,801,4745,44,78,NaN,NaN,211,NaN,1167,NaN,1862
top,BHEL,BHARAT HEAVY ELECTRICALS LTD.,PROMOTING EDUCATION,ANY OTHER RELATED ACTIVITY,SCHEDULE VII (II),NaN,NaN,CIPLA FOUNDATION,NaN,STATE AUTHORITIES,NaN,ALL STATES
freq,219,219,28,75,1706,NaN,NaN,25,NaN,58,NaN,505
mean,NaN,NaN,NaN,NaN,NaN,189.967053,129.277435,NaN,111.665804,NaN,112.014446,NaN
std,NaN,NaN,NaN,NaN,NaN,1290.548310,1019.837015,NaN,751.054812,NaN,684.429214,NaN
min,NaN,NaN,NaN,NaN,NaN,0.010000,0.050000,NaN,0.050000,NaN,0.050000,NaN
25%,NaN,NaN,NaN,NaN,NaN,4.000000,3.000000,NaN,2.127750,NaN,3.212000,NaN
50%,NaN,NaN,NaN,NaN,NaN,16.000000,11.200000,NaN,9.000000,NaN,10.600000,NaN
75%,NaN,NaN,NaN,NaN,NaN,68.500000,50.000000,NaN,37.375000,NaN,48.255000,NaN



--- Missing Values ---
Symbol                                                   0
Company                                                  0
Activity                                                 0
Sector                                                4932
Schedule                                                 0
Amount to be Spent as decided by Company (Rs.lacs)     599
Actual Amount Spent (Rs.lacs)                           71
Direct-Agency Name/s                                  4599
Direct-Amount (Rs.lacs)                               2294
Implementing Agency Name/s                            3515
Implementing Agency-Amount (Rs.lacs)                  2242
States (City/Town/District/Village)                     38
dtype: int64


# Section 3: Extract and Classify Locations
Parse the `'States (City/Town/District/Village)'` column to create a new location classification dataset. Values outside the brackets will be labeled as `'State'`, while the comma/slash-separated values inside the brackets will be labeled as `'Unclassified'` (to be manually classified into city/town/district/village later).

In [ ]:
import pandas as pd
import re

# Dictionary to hold unique locations and their classification
location_classification = {}

def extract_and_classify(loc_str):
    if pd.isna(loc_str) or not isinstance(loc_str, str):
        return
        
    loc_str = loc_str.strip()

    
    # Use finditer to capture multiple instances of "State (details)" or just "State" in a single string
    # group(1) matches everything up to the first parenthesis
    # group(2) matches contents inside parentheses
    matches = re.finditer(r'([^\(]+)(?:\(([^)]+)\))?', loc_str)
    
    for match in matches:
        state_part = match.group(1)
        details_part = match.group(2)
        
        if state_part:
            # Remove leading/trailing formatting characters like commas
            clean_state_part = state_part.strip(' ,')
            
            # Since there could be multiple standalone states separated by commas (e.g. "Delhi, Kerala"),
            # we can split the state_part itself by comma or slash.
            for s in re.split(r'[/,;]', clean_state_part):
                s = s.strip(' \t\n\r-|')
                if s:
                    # Anything outside brackets goes straight to 'State' class
                    location_classification[s] = 'State'
                    
        if details_part:
            # Everything inside brackets is split up by comma
            parts = [p.strip(' \t\n\r-|') for p in re.split(r'[,;&]', details_part) if p.strip()]
            for part in parts:
                if part:
                    # Always label inside-bracket values as 'Unclassified' if they haven't been stored as State already
                    if part not in location_classification:
                        location_classification[part] = 'Unclassified'

if not df.empty and 'States (City/Town/District/Village)' in df_csv.columns:
    print("Extracting and classifying locations...")
    
    # Apply the extraction function to every row in the column
    df_csv['States (City/Town/District/Village)'].apply(extract_and_classify)
    
    # Build the new dataset with 'Location' and 'Class' columns
    records = [{'Location': loc, 'Class': cls} for loc, cls in location_classification.items()]
    locations_df = pd.DataFrame(records)
    
    # Sort for better readability (States first, then alphabetical)
    locations_df = locations_df.sort_values(by=['Class', 'Location'], ascending=[True, True]).reset_index(drop=True)
    
    print("Extraction complete! Here is the Location Classification Dataset:")
    display(locations_df)
    print(f"Total unique locations extracted: {len(locations_df)}")
else:
    print("DataFrame empty or location column not found. Skipping extraction.")
    locations_df = pd.DataFrame()

Extracting and classifying locations...
Extraction complete! Here is the Location Classification Dataset:


,Location,Class
0,),State
1,AHMEDABAD,State
2,AHMEDABAD),State
3,ALL STATES,State
4,ANDAMAN & NICOBAR ISLANDS,State
...,...,...
2112,YAVATMAL,Unclassified
2113,YELAHANKA,Unclassified
2114,YETALA,Unclassified
2115,ZAHEERABAD,Unclassified


Total unique locations extracted: 2117


# Section 4: Save Classification Dataset
Save the new `locations_df` containing the `Location` and `Class` mappings into a new CSV file. This dataset can be easily updated or modeled upon later to classify the remaining specific locations.

In [4]:
if not locations_df.empty:
    reference_csv_path = 'Location_Classification.csv'
    locations_df.to_csv(reference_csv_path, index=False)
    print(f"Location classification dataset successfully saved to {reference_csv_path}")

Location classification dataset successfully saved to Location_Classification.csv


# Section 5: Intelligent Location Classification using Geopy
Use the `Nominatim` geocoder iteratively to inspect elements in `Location_Classification.csv`. Using the `addresstype` provided by the API, we can accurately assign categories like State, District, City, Town, or Village to these unclassified Indian locations. 

*Note: The geocoder is rate-limited to 1 request per second, so this cell might take a few minutes to process the entire dataset.*

In [5]:
import pandas as pd
import time
from geopy.geocoders import Nominatim

try:
    print("Loading locations dataset...")
    class_df = pd.read_csv('Location_Classification.csv')
    print(f"Loaded dataset with {len(class_df)} records.")
    
    geolocator = Nominatim(user_agent="india_csr_classifier")
    
    # Pre-defined known states and UTs to avoid unnecessary API lookups 
    # since we know these accurately already.
    known_states_uts = {
        "ANDAMAN & NICOBAR ISLANDS", "ANDHRA PRADESH", "ARUNACHAL PRADESH", "ASSAM", "BIHAR", 
        "CHANDIGARH", "CHHATTISGARH", "DADRA & NAGAR HAVELI AND DAMAN & DIU", "DELHI", "GOA", 
        "GUJARAT", "HARYANA", "HIMACHAL PRADESH", "JAMMU & KASHMIR", "JHARKHAND", "KARNATAKA", 
        "KERALA", "LADAKH", "MADHYA PRADESH", "MAHARASHTRA", "MANIPUR", "MEGHALAYA", "MIZORAM", 
        "NAGALAND", "ODISHA", "PUDUCHERRY", "PUNJAB", "RAJASTHAN", "SIKKIM", "TAMIL NADU", 
        "TELANGANA", "TRIPURA", "UTTAR PRADESH", "UTTARAKHAND", "WEST BENGAL"
    }

    def get_intelligent_class(row):
        loc = row['Location']
        current_class = row['Class']
        
        if pd.isna(loc) or not str(loc).strip():
            return "Not Found"
            
        clean_loc = str(loc).strip()
        
        # 1. Use manual override for States and commonly occurring Indian classifications
        if clean_loc.upper() in known_states_uts:
            return "State"
            
        # 2. Add sleep to respect Nominatim policy of max 1 req / second
        time.sleep(1.05)
        
        try:
            # Contextualize search heavily by appending India
            query = f"{clean_loc}, India"
            res = geolocator.geocode(query, addressdetails=True, timeout=10)
            
            if res and hasattr(res, 'raw') and 'addresstype' in res.raw:
                ctype = res.raw['addresstype'].lower()
                
                # Standardize OpenStreetMap categorization to requested Indian classes
                if ctype in ['state', 'union_territory', 'region', 'state_district']: 
                    # state_district sometimes acts as pure State/District boundary, but we'll map district below
                    if ctype == 'state_district': return "District"
                    return "State"
                if ctype in ['district', 'county', 'borough']: return "District"
                if ctype in ['city', 'municipality', 'city_district']: return "City"
                if ctype in ['town']: return "Town"
                if ctype in ['village', 'hamlet']: return "Village"
                
                # If it doesn't fall into the explicitly requested categories, mark as Town to be safe (since many unrecognized places are likely smaller towns or villages)
                return "Town"
                
            return "Not Found"
        except Exception as e:
            return "Not Found"

    print("Starting intelligent classification algorithm...")
    print("This will process 1 location per second via network API to respect usage policies. Please wait...")
    
    # Execute the algorithm across the dataset
    class_df['Intelligent_Class'] = class_df.apply(get_intelligent_class, axis=1)
    
    # Save the completely mirrored dataset!
    mirror_path = 'Location_Classification_Mirror.csv'
    class_df.to_csv(mirror_path, index=False)
    
    print(f"\nIntelligent processing complete! Mirror dataset saved successfully to '{mirror_path}'.")
    display(class_df.head(25))

except FileNotFoundError:
    print("Error: 'Location_Classification.csv' not found. Please run the previous sections of the notebook to generate it.")

Loading locations dataset...
Loaded dataset with 2116 records.
Starting intelligent classification algorithm...
This will process 1 location per second via network API to respect usage policies. Please wait...

Intelligent processing complete! Mirror dataset saved successfully to 'Location_Classification_Mirror.csv'.


,Location,Class,Intelligent_Class
0,AHMEDABAD,State,City
1,AHMEDABAD,State,City
2,ALL STATES,State,Not Found
3,ANDAMAN & NICOBAR ISLANDS,State,State
4,ANDHRA PRADESH,State,State
5,ARUNACHAL PRADESH,State,State
6,ASSAM,State,State
7,AZAD NAGAR,State,Town
8,BANASKANTHA DIST.,State,Not Found
9,BANASKANTHA DIST.,State,Not Found


In [7]:
import pandas as pd

try:
    class_df_mirror = pd.read_csv('Location_Classification_Mirror.csv')
    
    # Define a function to apply rule-based overrides
    def apply_keyword_overrides(row):
        loc = str(row['Location']).upper()
        current_class = row['Intelligent_Class']
        
        # Override rules
        if 'DIST.' in loc or 'DISTRICT' in loc:
            return 'District'
        elif 'VILLAGE' in loc:
            return 'Village'
        elif 'CITY' in loc:
            return 'City'
        elif 'TOWN' in loc:
            return 'Town'
        
        return current_class

    # Apply the overrides
    class_df_mirror['Intelligent_Class'] = class_df_mirror.apply(apply_keyword_overrides, axis=1)
    
    # Save the updated classification mapping
    class_df_mirror.to_csv('Location_Classification_Mirror.csv', index=False)
    print("Keyword-based overrides applied successfully! Mirror dataset updated.")
    
    # Print updated value counts
    print("\n--- Updated Intelligent Classification Counts ---")
    display(class_df_mirror['Intelligent_Class'].value_counts())
    
except FileNotFoundError:
    print("Error: 'Location_Classification_Mirror.csv' not found.")

Keyword-based overrides applied successfully! Mirror dataset updated.

--- Updated Intelligent Classification Counts ---


Intelligent_Class
District     628
Town         562
Not Found    345
Village      278
City         266
State         37
Name: count, dtype: int64

# Section 7: Fallback Heuristic Classification for Remaining "Not Found" Locations
For the remaining locations that `geopy` couldn't resolve, we can apply Indian geographic suffix and keyword heuristics. For example, locations containing `-gaon`, `-wadi`, or `-halli` are typically Villages, while `-nagar`, `-pur`, or `-mandal` often denote Towns.

In [15]:
import pandas as pd

try:
    print("Loading mirror dataset for final heuristic classification...")
    fallback_df = pd.read_csv('Location_Classification_Mirror.csv')
    
    # Common Indian suffixes/words for villages
    village_keywords = ['GAON', 'WADI', 'PALLY', 'HALLI', 'KUPPAM', 'PADA', 'GUDA', 'GRAM', 'PANCHAYAT', 'THANDA', 'CHERRY', 'VILLAGE', 'KHERA', 'PURAM']
    
    # Common Indian suffixes/words for towns
    town_keywords = ['NAGAR', 'PUR', 'ABAD', 'BAGH', 'VIHAR', 'KHAND', 'PET', 'TALUKA', 'TEHSIL', 'MANDAL', 'CITY', 'TOWN', 'URBAN', 'KOTA', 'NAGRI', 'COLONY', 'ESTATE']
    
    # Other indicators like Roads, Crossings, etc., map to Town/Locality (but we'll keep it strictly 'Town' or 'Village')
    landmark_keywords = ['SCHOOL', 'HOSPITAL', 'CLINIC', 'ROAD', 'MARG', 'CROSSING', 'AREA', 'SLUM', 'VALLEY', 'PLANT', 'MINES', 'MACHINE']

    def apply_heuristics(row):
        loc = str(row['Location']).upper()
        current_class = row['Intelligent_Class']
        
        # Only process if it is still classified as 'Not Found'
        if current_class != 'Not Found':
            return current_class
            
        # 1. Check for Village keywords
        if any(kw in loc for kw in village_keywords):
            return 'Village'
            
        # 2. Check for Town keywords
        if any(kw in loc for kw in town_keywords):
            return 'Town'
            
        # 3. Landmarks, Institutions, or local areas map to Town
        if any(kw in loc for kw in landmark_keywords):
            return 'Town'
            
        # 4. Basic fallback - if it contains '&', it's usually an area description
        if '&' in loc:
            return 'Town'
            
        # If absolutely nothing hits, mark as Town to be safe (since many unrecognized places are likely smaller towns or villages)
        return 'Town'

    # Apply the heuristics to the remaining rows
    before_not_found = (fallback_df['Intelligent_Class'] == 'Not Found').sum()
    
    fallback_df['Intelligent_Class'] = fallback_df.apply(apply_heuristics, axis=1)
    
    after_not_found = (fallback_df['Intelligent_Class'] == 'Not Found').sum()
    
    # Save the final mapping
    fallback_df.to_csv('Location_Classification_Mirror.csv', index=False)
    
    print(f"Heuristics applied. Reduced 'Not Found' count from {before_not_found} to {after_not_found}!")
    
    # Print updated value counts
    print("\n--- Final Intelligent Classification Counts ---")
    display(fallback_df['Intelligent_Class'].value_counts())
    
except FileNotFoundError:
    print("Error: 'Location_Classification_Mirror.csv' not found.")

Loading mirror dataset for final heuristic classification...
Heuristics applied. Reduced 'Not Found' count from 205 to 0!

--- Final Intelligent Classification Counts ---


Intelligent_Class
Town        879
District    628
Village     304
City        267
State        38
Name: count, dtype: int64

# Section 8: Unravel and Flatten the Original Dataset
Using the mapped classifications from `Location_Classification_Mirror.csv`, we will now "unravel" the original `CSR_activities_2014-15.csv` dataset. This will explode the nested `States (City/Town/District/Village)` column into individual rows for each specific state and sub-location, and assign the intelligent class to each entry.

In [17]:
import pandas as pd
import re

try:
    print("Loading original datasets and classification mappings...")
    # Load original data and classification mapping
    orig_df = pd.read_csv('CSR_activities_2014-15.csv')
    mapping_df = pd.read_csv('Location_Classification_Mirror.csv')

    # Create a mapping dictionary: Location -> Intelligent_Class
    loc_to_class = dict(zip(mapping_df['Location'].str.strip(), mapping_df['Intelligent_Class']))

    exploded_records = []
    
    # Iterate through original rows
    for idx, row in orig_df.iterrows():
        loc_str = row['States (City/Town/District/Village)']
        
        # Base dictionary with all other row info
        base_record = row.to_dict()
        # Remove the nested column as we'll replace it with structured ones
        del base_record['States (City/Town/District/Village)']
        
        if pd.isna(loc_str) or not isinstance(loc_str, str):
            rec = base_record.copy()
            rec['Resolved_State'] = None
            rec['Resolved_Location'] = None
            rec['Location_Class'] = None
            exploded_records.append(rec)
            continue
            
        loc_str = loc_str.strip()
        matches = re.finditer(r'([^\(]+)(?:\(([^)]+)\))?', loc_str)
        
        found_any = False
        
        for match in matches:
            state_part = match.group(1)
            details_part = match.group(2)
            
            # Extract States
            states = []
            if state_part:
                clean_state_part = state_part.strip(' ,')
                for s in re.split(r'[/,;]', clean_state_part):
                    s = s.strip(' \t\n\r-|')
                    if s:
                        states.append(s)
            
            # Extract Sub-locations (details inside parentheses)
            sub_locs = []
            if details_part:
                parts = [p.strip(' \t\n\r-|') for p in re.split(r'[,;&]', details_part) if p.strip()]
                for p in parts:
                    if p:
                        sub_locs.append(p)
            
            # Record association logic
            if states and sub_locs:
                # E.g., MAHARASHTRA (MUMBAI, BOISAR)
                for s in states:
                    for sl in sub_locs:
                        rec = base_record.copy()
                        rec['Resolved_State'] = s
                        rec['Resolved_Location'] = sl
                        cls = loc_to_class.get(sl, "Not Found")
                        rec['Location_Class'] = cls
                        exploded_records.append(rec)
                        found_any = True
            elif states and not sub_locs:
                # E.g., "ALL STATES" or pure "GUJARAT"
                for s in states:
                    rec = base_record.copy()
                    rec['Resolved_State'] = s
                    rec['Resolved_Location'] = s
                    cls = loc_to_class.get(s, "State")  # Usually 'State'
                    rec['Location_Class'] = cls
                    exploded_records.append(rec)
                    found_any = True
            elif sub_locs and not states: 
                # Failsafe logic
                for sl in sub_locs:
                    rec = base_record.copy()
                    rec['Resolved_State'] = "Unknown"
                    rec['Resolved_Location'] = sl
                    cls = loc_to_class.get(sl, "Not Found")
                    rec['Location_Class'] = cls
                    exploded_records.append(rec)
                    found_any = True
                    
        # If the regex parsing missed completely, dump the raw string
        if not found_any:
            rec = base_record.copy()
            rec['Resolved_State'] = None
            rec['Resolved_Location'] = loc_str
            rec['Location_Class'] = "Not Found"
            exploded_records.append(rec)

    # Convert the list of exploded records back to a dataframe
    exploded_df = pd.DataFrame(exploded_records)
    
    # Save the unravelled dataset
    output_file = 'CSR_activities_Unravelled.csv'
    exploded_df.to_csv(output_file, index=False)
    
    print(f"Unravelled dataset successfully created and saved to '{output_file}'.")
    print(f"Original records: {len(orig_df)}")
    print(f"Expanded records: {len(exploded_df)}")
    
    print("\n--- Sample of Expanded Dataset ---")
    # Show the last 3 columns to specifically highlight the unraveled logic
    display(exploded_df.iloc[:, [1, 2, 9, -3, -2, -1]].head(15))
    
except FileNotFoundError as e:
    print(f"Error loading required files: {e}")

Loading original datasets and classification mappings...
Unravelled dataset successfully created and saved to 'CSR_activities_Unravelled.csv'.
Original records: 5136
Expanded records: 12485

--- Sample of Expanded Dataset ---


,Company,Activity,Implementing Agency Name/s,Resolved_State,Resolved_Location,Location_Class
0,3M INDIA LTD.,DONATION TOWARDS JAMMU & KASHMIR DISASTER RELI...,NaN,JAMMU & KASHMIR,JAMMU & KASHMIR,State
1,3M INDIA LTD.,EDUCATION: MOBILE SCIENCE LAB INITIATIVE FOR D...,NaN,KARNATAKA,ANEKAL,City
2,3M INDIA LTD.,EDUCATION: MOBILE SCIENCE LAB INITIATIVE FOR D...,NaN,MAHARASHTRA,RANJANGAON,Village
3,3M INDIA LTD.,EDUCATION: MOBILE SCIENCE LAB INITIATIVE FOR D...,NaN,MAHARASHTRA,PUNE,City
4,63 MOONS TECHNOLOGIES LTD.,A) DN POLYTECHNIC EDUCATION TR UST (ENTREPRENE...,NaN,ALL STATES,ALL STATES,State
5,63 MOONS TECHNOLOGIES LTD.,B) SHRIMATI SAVITA MAHILA UTKARSH SEVA SANGH (...,NaN,ALL STATES,ALL STATES,State
6,63 MOONS TECHNOLOGIES LTD.,C) JAYBHARTI FOUNDATION 50.00 (HEALTH AWARENES...,NaN,ALL STATES,ALL STATES,State
7,AARTI DRUGS LTD.,EDUCATION & SKILL DEVELOPMENT,NaN,GUJARAT,KACHCHH,District
8,AARTI DRUGS LTD.,EDUCATION & SKILL DEVELOPMENT,NaN,MAHARASHTRA,MUMBAI,City
9,AARTI DRUGS LTD.,EDUCATION & SKILL DEVELOPMENT,NaN,MAHARASHTRA,PALGHAR,District
